# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
   
ANS: No. One hundred successful demonstrations do not prove that the robot can perform the task across every home environment.     They only demonstrate success under the specific conditions tested. Changes in furniture arrangements, obstacles, mug fill levels, or Eva's position may affect the robot's performance. Therefore, broader competence requires testing across varied conditions and evaluating both successes and failures.
Confidence Rating: 5/5.Repeated success provides evidence of performance under tested conditions, but does not guarantee reliable performance in unfamiliar environments.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
 
 ANS: When the nearly full mug is carried through a turn, the liquid may move due to the robot's motion and spill over the rim because there is less empty space inside the mug. The original successful run left untested the assumption that the same carrying trajectory would remain safe for different fill levels. Therefore, the robot must consider the fill level and adjust its movement to prevent spilling.
     
3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.

ANS: The robot should ask Eva whether she accepts the white mug instead of proceeding immediately. Since the blue mug is unavailable, the robot cannot assume that the white mug is an acceptable substitute. Detecting the white mug only confirms its availability, not its suitability for Eva's request.
The single item of evidence required is Eva's explicit confirmation that she accepts the white mug.
The robot should resolve the uncertainty through confirmation before committing to the action. It must still verify the feasibility and safety of the delivery before execution.Required evidence: Eva must explicitly confirm that she accepts the white mug as a substitute for the blue mug.
     
4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
 
 ANS: T1 – Explain: Was the coffee delivered without spilling because the robot turned slowly, or because the mug was only half-full? How did the turning speed and the amount of liquid contribute to the result?
   T2 – Predict: If the same robot carries the same mug, which is now nearly full, along the same route and makes the same turn, will the coffee spill? Why might the changed fill level affect the liquid's movement?
   T3 – Design: Would using a different gripper or placing the mug on a tray improve the safety of the delivery? How should the robot adapt its carrying speed, turning motion, or trajectory to reduce the risk of spilling?
   T4 – Bound: Under which conditions can the robot reliably deliver coffee without spilling? What combinations of mug types, liquid fill levels, carrying speeds, routes, and robot bodies are within its demonstrated capabilities, and where might it fail?
   T5 – Check: What evidence should be recorded to verify that the robot completed the delivery successfully? Does the record show Eva's acceptance of the mug, the robot's actual movements, whether any coffee was spilled, and whether Eva received the coffee?
   Changed condition: T2 addresses the nearly full mug. A video showing the robot successfully carrying a half-full mug cannot establish whether it will succeed with a nearly full mug. The robot must make a prediction before the changed episode and compare it with the actual observations afterwards. 
 

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
    
 ANS: To handle Eva's request, the robot requires the following functional modules, integrated through the Manipulation Executive and the Body–World System.
1. Interpreting & Grounding: Understand Eva's request and identify the intended coffee, mug, and goal.
2. Perceiving: Use sensors to identify objects, estimate the mug's fill level, and determine Eva's position.
3. Proposing: Generate possible actions, such as selecting the white mug, grasping it, and carrying it to Eva.
4. Predicting: Use the world model or digital twin to predict the consequences of actions, such as possible spilling or collisions.
5. Warranting: Check whether the proposed action satisfies grounding, feasibility, safety, monitoring, repair, and recording requirements before execution.
6. Acting & Regulating: Control the robot's movements, monitor execution, and adjust actions when the situation changes.
7. Recording & Justifying: Record the request, assumptions, decisions, observations, and outcomes for later evaluation.

Overall flow:

   Interpreting  & Grounding → Perceiving → Proposing → Predicting → Warranting → Acting & Regulating → Recording & Justifying.

   The Warranting is important because it prevents an unverified action proposal from becoming a physical commitment.


6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.
  
  ANS: If Eva stands up while the robot is carrying the coffee, the robot should pause its current movement, observe Eva's new position, and re-plan the route and handover position. It should not continue following the original path without reassessment.

The potential failure mechanism is that Eva's movement may make the original handover position unsuitable or create a collision risk. The coffee may also spill if the robot makes sudden movements while adjusting its path.

Evidence Criterion: Before committing to the revised path, the robot must have sufficient evidence that Eva's current position is correctly identified, the new route is clear and reachable, the mug can be carried safely, and the revised handover is feasible. All six warrant checks—grounding, feasibility, safety, monitoring, repairability, and recordability—must satisfy their declared criteria.
Conclusion: The robot preserves Eva's goal but adapts its actions to the changed situation. It should only proceed when the revised action passes the commitment check.

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [11]:
#!/usr/bin/env python3
"""Embodied Cognitive Architecture
"""

from dataclasses import dataclass, field
from typing import Dict, Any, Tuple, Optional, List
import random
import time

@dataclass(frozen=True)
class GoalSpecification:
    """Formalized, concrete goal specification derived via semantic interpretation."""
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float

@dataclass
class ParameterPacket:
    """System 1 (The Proposer): Flexible sampler proposing execution parameters."""
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    contact_depth_m: float = 0.01
    source_generator: str = "stochastic_sampler"
    estimated_battery_drain_percent: float = 16.0


class Proposer:
    """Generates candidate action parameters using uniform sampling."""

    def __init__(self):
        self.min_velocity = 0.1
        self.max_velocity = 0.6
        self.min_tilt_rate = 0.01
        self.max_tilt_rate = 0.15

    def propose(self) -> ParameterPacket:
        velocity = random.uniform(
            self.min_velocity,
            self.max_velocity
        )

        tilt_rate = random.uniform(
            self.min_tilt_rate,
            self.max_tilt_rate
        )

        grasp_pose = (
            random.uniform(0.3, 0.5),
            random.uniform(0.1, 0.2),
            random.uniform(0.2, 0.4)
        )

        return ParameterPacket(
            grasp_pose=grasp_pose,
            max_velocity_mps=velocity,
            tilt_rate_radps=tilt_rate,
            source_generator="uniform_sampler"
        )


@dataclass
class Observation:
    """Raw, noisy sensor measurements at time t (distinct from internal belief)."""
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str

@dataclass
class BeliefState:
    """The robot's current internal belief state (updated via perception)."""
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    # Real-time environmental parameters maintained internally
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"  # 'low', 'medium', 'high'
    preferred_cup: str = "blue_ceramic_mug"
    battery_level_percent: float = 15.0


class InterpretationSystem:
    """Uses a semantic knowledge base to map vague instructions and context 
    into precise physical goal parameters (GoalSpecification).
    """
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        print(f"\n[INTERPRETATION] Analyzing instruction: '{raw_utterance}' in context of Eva (Position: {belief.eva_position}, Room Busyness: {belief.room_busyness})")
        
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        
        busyness_factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        max_nav_vel = 1.2 * busyness_factor
        
        is_urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        time_frame = 30.0 if is_urgent else 60.0
        target_volume = 15.0 if not is_urgent else 12.0
        
        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=target_volume,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=max_nav_vel,
            time_frame_seconds=time_frame
        )
        
        print(f" -> Derived Goal: Object={goal.target_object}, MaxForce={goal.max_allowable_force_n}N, MaxNavVel={goal.max_nav_velocity_mps}m/s, TimeFrame={goal.time_frame_seconds}s")
        return goal


class SensingSystem:
    """Simulates real-time noisy hardware sensors emitting raw observations."""
    def __init__(self):
        pass

    def sense(self, true_ambient_temp: float, true_friction: float, eva_pos: str, busyness: str) -> Observation:
        noisy_temp = true_ambient_temp + random.uniform(-0.5, 0.5)
        noisy_friction = true_friction + random.uniform(-0.05, 0.05)
        
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(noisy_temp, 2),
            measured_friction=round(noisy_friction, 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    """Estimator module that updates the internal BeliefState using raw Observations."""
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    """System 2 (Semantic Digital Twin): Imperfect models with sim-to-real epistemic gaps."""
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    

    def predict_properties(
        self, params: ParameterPacket, goal: GoalSpecification
    ) -> Dict[str, float]:

        # Stage 1: Normal contact stiffness
        stiffness_coefficient = (
            12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        )

        # Stage 2: Increased stiffness after bottoming-out
        if params.contact_depth_m > 0.02:
            stiffness_coefficient *= 1.8

        # Calculate predicted contact force
        predicted_force = (
            stiffness_coefficient * (params.max_velocity_mps ** 1.5)
        )

        # Temperature-dependent fluid viscosity
        reference_temperature_c = 20.0
        thermal_degradation_coefficient = 0.01

        temperature_difference = (
            self.belief.ambient_temperature_c - reference_temperature_c
        )

        effective_viscosity = self.belief.fluid_viscosity_pas * (
            1.0 - thermal_degradation_coefficient * temperature_difference
        )

        effective_viscosity = max(effective_viscosity, 0.01)

        radius = 0.05
        wave_height_m = (
            params.tilt_rate_radps * radius
        ) / effective_viscosity

        predicted_wave = wave_height_m * 1000.0

        # Sim-to-real epistemic prediction error
        epistemic_force_error = random.uniform(-0.2, 0.3)
        epistemic_wave_error = random.uniform(-0.1, 0.2)

        return {
            "predicted_force_n": round(
                max(0.0, predicted_force + epistemic_force_error), 2
            ),
            "predicted_wave_mm": round(
                max(0.0, predicted_wave + epistemic_wave_error), 2
            ),
            "predicted_nav_velocity": min(
                params.max_velocity_mps, goal.max_nav_velocity_mps
            ),
        }



class WarrantGate:
    """Validates predicted properties against the interpreted GoalSpecification."""
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification, belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (predictions["predicted_force_n"] <= goal.max_allowable_force_n) and 
                    (predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm) and
                    (predictions["predicted_nav_velocity"] <= goal.max_nav_velocity_mps),
            "energetically_viable": (params.estimated_battery_drain_percent < 15.0 
                                     and params.estimated_battery_drain_percent
                                     <= belief.battery_level_percent
                                    ),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled
        }
        
        all_passed = all(checks.values())
        failed_criterion = next((name for name, passed in checks.items() if not passed), None)
        
        return {
            "passed": all_passed,
            "failed_criterion": failed_criterion,
            "predictions": predictions,
            "warrant_checks": checks
        }


class ExecutionEngine:
    """Manages physical execution and constructs the epistemic audit trail."""
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification, warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[ACTUATOR] Executing motion in physical world: v_max={params.max_velocity_mps} m/s")
        time.sleep(0.1)
        
        # Real-world divergence check
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)
        
        spill_occurred = (actual_force > goal.max_allowable_force_n) or (actual_wave > goal.max_slosh_wave_mm)
        status = "FAILURE" if spill_occurred else "SUCCESS"
        
        execution_outcome = {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill_occurred else goal.target_volume_ml - 3.0,
            "spill_occurred": spill_occurred,
            "status": status
        }
        return execution_outcome
    def record_trace(
        self, raw_command, goal, belief, proposal,
        warrant, observation, committed=False
):
        trace = {
            "asked": {
                "raw_utterance": raw_command,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
            }
        },

                "belief": {
                    "eva_position": belief.eva_position,
                    "room_busyness": belief.room_busyness,
                    "ambient_temp_c": belief.ambient_temperature_c,
                    "surface_friction": belief.surface_friction_coefficient,
                    "battery_level_percent": belief.battery_level_percent
                },

                "committed": {
                    "action_approved": committed,
                    "parameters": {
                        "max_velocity_mps": proposal.max_velocity_mps,
                        "tilt_rate_radps": proposal.tilt_rate_radps,
                        "contact_depth_m": proposal.contact_depth_m
                    },
                    "warrant_verdict": warrant
                },

                "observed": {
                    "timestamp": observation.timestamp,
                    "measured_temp_c": observation.measured_temperature_c,
                    "measured_friction": observation.measured_friction,
                    "eva_position": observation.eva_position,
                    "room_busyness": observation.room_busyness
                },

                "execution_outcome": None
            }

        self.audit_trail.append(trace)
        
def audit_analyzer(audit_trail):
    approved = 0
    rejected = 0
    failed_criteria = []
    force_errors = []

    for trace in audit_trail:
        committed = trace["committed"]
        warrant = committed["warrant_verdict"]

        if committed["action_approved"]:
            approved += 1
        else:
            rejected += 1

        for criterion, passed in warrant["warrant_checks"].items():
            if not passed:
                failed_criteria.append(criterion)

        outcome = trace.get("execution_outcome")

        if outcome is not None:
            predicted_force = warrant["predictions"]["predicted_force_n"]
            actual_force = outcome["actual_force_n"]
            force_errors.append(abs(actual_force - predicted_force))

    counts = {
        criterion: failed_criteria.count(criterion)
        for criterion in set(failed_criteria)
    }

    if counts:
        highest_count = max(counts.values())
        most_frequent_failure = [
            criterion
            for criterion, count in counts.items()
            if count == highest_count
        ]
    else:
        most_frequent_failure = []

    import statistics

    average_force_variance = (
        round(statistics.pvariance(force_errors), 3)
        if force_errors else None
    )

    return {
        "approved_proposals": approved,
        "rejected_proposals": rejected,
        "most_frequent_failed_criterion": most_frequent_failure,
        "failed_criterion_frequency": counts,
        "average_force_prediction_error_variance": average_force_variance
    }
# --- Execution Demonstration ---
if __name__ == "__main__":
    knowledge_base = {
        "cups": {
            "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
            "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
        },
        "room_busyness_limits": {
            "low": 1.0,
            "medium": 0.6,
            "high": 0.3
        }
    }

    # 1. Initialize systems
    belief_state = BeliefState(room_busyness="medium", preferred_cup="blue_ceramic_mug")
    sensing = SensingSystem()
    perception = PerceptionSystem(belief_state)
    interpreter = InterpretationSystem(knowledge_base)
    sdt = SemanticDigitalTwin(belief_state)
    warrant_gate = WarrantGate(sdt)
    executive = ExecutionEngine()
    
    print("=== STEP 1: Sensing & Perception Update ===")
    raw_observation = sensing.sense(
        true_ambient_temp=21.5, 
        true_friction=0.45, 
        eva_pos="living_room", 
        busyness="medium"
    )
    perception.update(raw_observation)
    print(f"Raw Sensor Observation : {raw_observation}")
    print(f"Updated Internal Belief : Temp={belief_state.ambient_temperature_c}°C, Friction={belief_state.surface_friction_coefficient}")
    
    print("\n=== STEP 2: Instruction Interpretation ===")
    raw_command = "Bring me my coffee fast!"
    goal = interpreter.interpret(raw_command, belief_state)
    
    print("\n=== STEP 3: System 1 Proposer Generates Parameters ===")
    proposal_v1 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.8, 
        tilt_rate_radps=0.2
    )
    print(f"Proposer suggests: v_max = {proposal_v1.max_velocity_mps} m/s")
    
    print("\n=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===")
    warrant_v1 = warrant_gate.validate(proposal_v1, goal, belief_state)
    print(f"SDT Predictions: Nav-Vel = {warrant_v1['predictions']['predicted_nav_velocity']} m/s")
    print(f"Warrant Checks  : {warrant_v1['warrant_checks']}")
    print(f"Gate Verdict    : REJECTED (Failed on: {warrant_v1['failed_criterion']})")
    executive.record_trace(
    raw_command, goal, belief_state,
    proposal_v1, warrant_v1, raw_observation,
    committed=False
    )
    
    
    print("\n=== STEP 5: Proposer Resamples (Attempt 2) ===")
    proposal_v2 = ParameterPacket(
        grasp_pose=(0.45, 0.12, 0.30),
        max_velocity_mps=0.45, 
        tilt_rate_radps=0.08
    )
    
    warrant_v2 = warrant_gate.validate(proposal_v2, goal, belief_state)
    print(f"SDT Predictions: Force = {warrant_v2['predictions']['predicted_force_n']} N, Wave = {warrant_v2['predictions']['predicted_wave_mm']} mm")
    print(f"Warrant Checks  : {warrant_v2['warrant_checks']}")
    if warrant_v2["passed"]:
        print("Gate Verdict    : APPROVED")
    else:
        print(
            f"Gate Verdict    : REJECTED "
            f"(Failed on: {warrant_v2['failed_criterion']})"
        )
    executive.record_trace(
        raw_command, goal, belief_state,
        proposal_v2, warrant_v2, raw_observation,
        committed=False
    )
    print("\n=== STEP 6: Physical Execution & Epistemic Audit Trail ===")
    if warrant_v2["passed"]:
        final_outcome = executive.execute_action(
            proposal_v2, goal, warrant_v2
    )

        executive.audit_trail[-1]["committed"]["action_approved"] = True
        executive.audit_trail[-1]["execution_outcome"] = final_outcome

        print(f"Execution Outcome: {final_outcome}")

    print(f"\n--- COMPLETE EPISODIC AUDIT TRAIL ({len(executive.audit_trail)} Entries) ---")
    for idx, trace in enumerate(executive.audit_trail, 1):
        status = trace['committed']['action_approved']
        outcome = trace.get('execution_outcome')
        outcome_str = f" | Outcome: {outcome['status']} (Spill: {outcome['spill_occurred']})" if outcome else ""
        print(f"Entry #{idx} | Approved: {status}{outcome_str} | Command: '{trace['asked']['raw_utterance']}'")

=== STEP 1: Sensing & Perception Update ===
Raw Sensor Observation : Observation(timestamp=1793612743.359301, measured_temperature_c=21.22, measured_friction=0.4, eva_position='living_room', room_busyness='medium')
Updated Internal Belief : Temp=21.22°C, Friction=0.4

=== STEP 2: Instruction Interpretation ===

[INTERPRETATION] Analyzing instruction: 'Bring me my coffee fast!' in context of Eva (Position: living_room, Room Busyness: medium)
 -> Derived Goal: Object=blue_ceramic_mug, MaxForce=4.5N, MaxNavVel=0.72m/s, TimeFrame=30.0s

=== STEP 3: System 1 Proposer Generates Parameters ===
Proposer suggests: v_max = 0.8 m/s

=== STEP 4: System 2 SDT & Warrant Gate Validation (Attempt 1) ===
SDT Predictions: Nav-Vel = 0.72 m/s
Warrant Checks  : {'grounded': True, 'feasible': True, 'safe': False, 'energetically_viable': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate Verdict    : REJECTED (Failed on: safe)

=== STEP 5: Proposer Resamples (Attempt 2) ===
SDT Predicti

In [12]:
print("Audit trail length:", len(executive.audit_trail))
audit_analyzer(executive.audit_trail)

print("record_trace:", hasattr(executive, "record_trace"))
print("audit_analyzer:", callable(audit_analyzer))

Audit trail length: 2
record_trace: True
audit_analyzer: True


In [13]:
proposer = Proposer()

for i in range(5):
    proposal = proposer.propose()
    print(f"Proposal {i+1}: {proposal}")


Proposal 1: ParameterPacket(grasp_pose=(0.4803722803273425, 0.12391719909947295, 0.3973054713489632), max_velocity_mps=0.47907487368077784, tilt_rate_radps=0.03965895925828393, contact_depth_m=0.01, source_generator='uniform_sampler', estimated_battery_drain_percent=16.0)
Proposal 2: ParameterPacket(grasp_pose=(0.4582232993145951, 0.12426523081686054, 0.2386359106036335), max_velocity_mps=0.3501303648576811, tilt_rate_radps=0.03268659664225894, contact_depth_m=0.01, source_generator='uniform_sampler', estimated_battery_drain_percent=16.0)
Proposal 3: ParameterPacket(grasp_pose=(0.36097415385291337, 0.19821714237210775, 0.24856383392866752), max_velocity_mps=0.18584662489864226, tilt_rate_radps=0.11896880537193989, contact_depth_m=0.01, source_generator='uniform_sampler', estimated_battery_drain_percent=16.0)
Proposal 4: ParameterPacket(grasp_pose=(0.4559324729006615, 0.180004105149909, 0.3160966121295795), max_velocity_mps=0.24797272411502894, tilt_rate_radps=0.09760206387469339, conta

In [14]:

# Task 2.2: Aggressive contact test

aggressive_proposal = ParameterPacket(
    grasp_pose=(0.45, 0.12, 0.30),
    max_velocity_mps=0.35,
    tilt_rate_radps=0.1,
    contact_depth_m=0.025
)

# Validate through the Warrant Gate
aggressive_result = warrant_gate.validate(
    aggressive_proposal, goal, belief_state
)

print("=== TASK 2.2: AGGRESSIVE CONTACT TEST ===")
print("Predicted force:",
      aggressive_result["predictions"]["predicted_force_n"], "N")

print("Warrant checks:",
      aggressive_result["warrant_checks"])

print("Gate passed:", aggressive_result["passed"])
print("Failed criterion:",
      aggressive_result["failed_criterion"])

if not aggressive_result["passed"]:
    print("REJECTED: Physical execution must not proceed.")
else:
    print("APPROVED: Proposal may proceed to execution.")


=== TASK 2.2: AGGRESSIVE CONTACT TEST ===
Predicted force: 4.86 N
Warrant checks: {'grounded': True, 'feasible': True, 'safe': False, 'energetically_viable': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate passed: False
Failed criterion: safe
REJECTED: Physical execution must not proceed.


In [15]:
# Q3.2: Compare wave height at different temperatures

test_tilt_rate = 0.15

# Keep the same tilt rate and compare two temperatures
original_temperature = belief_state.ambient_temperature_c

params = ParameterPacket(
    grasp_pose=(0.45, 0.12, 0.30),
    max_velocity_mps=0.45,
    tilt_rate_radps=test_tilt_rate,
    contact_depth_m=0.01
)

# Prediction at reference temperature
belief_state.ambient_temperature_c = 20.0
reference_result = sdt.predict_properties(params, goal)

# Prediction at higher temperature
belief_state.ambient_temperature_c = 28.0
hot_result = sdt.predict_properties(params, goal)

# Restore original temperature
belief_state.ambient_temperature_c = original_temperature

print("=== TASK 3.2: TEMPERATURE AND FLUID SLOSHING ===")
print("Tilt rate:", test_tilt_rate, "rad/s")
print("Wave at 20°C:",
      reference_result["predicted_wave_mm"], "mm")
print("Wave at 28°C:",
      hot_result["predicted_wave_mm"], "mm")

=== TASK 3.2: TEMPERATURE AND FLUID SLOSHING ===
Tilt rate: 0.15 rad/s
Wave at 20°C: 6.35 mm
Wave at 28°C: 6.96 mm


In [16]:
# TASK 4.2: LOW-BATTERY TEST

belief_state.battery_level_percent = 15.0

test_proposal = ParameterPacket(
    grasp_pose=(0.45, 0.12, 0.30),
    max_velocity_mps=0.45,
    tilt_rate_radps=0.08,
    contact_depth_m=0.01,
    estimated_battery_drain_percent=16.0
)

energy_result = warrant_gate.validate(
    test_proposal, goal, belief_state
)

print("=== TASK 4.2: LOW-BATTERY TEST ===")
print("Battery level:", belief_state.battery_level_percent, "%")
print("Warrant checks:", energy_result["warrant_checks"])
print("Gate passed:", energy_result["passed"])
failed_criteria = [
    name
    for name, passed in energy_result["warrant_checks"].items()
    if not passed
]

print("All failed criteria:", failed_criteria)

if not energy_result["warrant_checks"]["energetically_viable"]:
    print("REJECTED: Redirecting robot to recharge.")
    print("Coffee delivery postponed until sufficient battery is available.")

=== TASK 4.2: LOW-BATTERY TEST ===
Battery level: 15.0 %
Warrant checks: {'grounded': True, 'feasible': True, 'safe': False, 'energetically_viable': False, 'monitorable': True, 'repairable': True, 'recordable': True}
Gate passed: False
All failed criteria: ['safe', 'energetically_viable']
REJECTED: Redirecting robot to recharge.
Coffee delivery postponed until sufficient battery is available.


In [17]:
# Verify the ExecutionEngine created in the main execution cell
print("Audit trail length:", len(executive.audit_trail))
print("record_trace:", hasattr(executive, "record_trace"))
print("audit_analyzer:", callable(audit_analyzer))

Audit trail length: 2
record_trace: True
audit_analyzer: True


In [18]:
# Verify the existing ExecutionEngine without resetting its audit trail
print("Audit trail length:", len(executive.audit_trail))
print("record_trace:", hasattr(executive, "record_trace"))
print("audit_analyzer:", callable(audit_analyzer))

Audit trail length: 2
record_trace: True
audit_analyzer: True


In [19]:
print("Current audit trail length:", len(executive.audit_trail))

for i, trace in enumerate(executive.audit_trail, 1):
    print(
        i,
        trace["asked"]["raw_utterance"],
        trace["committed"]["action_approved"]
    )

Current audit trail length: 2
1 Bring me my coffee fast! False
2 Bring me my coffee fast! False


In [20]:
print("Audit trail length:", len(executive.audit_trail))
audit_analyzer(executive.audit_trail)

Audit trail length: 2


{'approved_proposals': 0,
 'rejected_proposals': 2,
 'most_frequent_failed_criterion': ['safe', 'energetically_viable'],
 'failed_criterion_frequency': {'safe': 2, 'energetically_viable': 2},
 'average_force_prediction_error_variance': None}

In [21]:
analysis = audit_analyzer(executive.audit_trail)

print("Raw analyzer result:", analysis)
print("Audit trail length:", len(executive.audit_trail))

Raw analyzer result: {'approved_proposals': 0, 'rejected_proposals': 2, 'most_frequent_failed_criterion': ['safe', 'energetically_viable'], 'failed_criterion_frequency': {'safe': 2, 'energetically_viable': 2}, 'average_force_prediction_error_variance': None}
Audit trail length: 2


In [22]:
print("=== TASK 5.1: AUDIT ANALYSIS ===")

analysis = audit_analyzer(executive.audit_trail)

print("Approved:", analysis["approved_proposals"])
print("Rejected:", analysis["rejected_proposals"])
print("Most frequent failed criteria:",
      analysis["most_frequent_failed_criterion"])
print("Failure frequency:", analysis["failed_criterion_frequency"])
print("Average force prediction error variance:",
      analysis["average_force_prediction_error_variance"])

=== TASK 5.1: AUDIT ANALYSIS ===
Approved: 0
Rejected: 2
Most frequent failed criteria: ['safe', 'energetically_viable']
Failure frequency: {'safe': 2, 'energetically_viable': 2}
Average force prediction error variance: None


In [23]:
print("=== TASK 5.2: EPISTEMIC AUDIT TRACE ===")

for i, trace in enumerate(executive.audit_trail, 1):
    print(f"\nEntry {i}")
    print("Asked:", trace.get("asked"))
    print("Belief:", trace.get("belief"))
    print("Committed:", trace.get("committed"))
    print("Observed:", trace.get("observed"))

=== TASK 5.2: EPISTEMIC AUDIT TRACE ===

Entry 1
Asked: {'raw_utterance': 'Bring me my coffee fast!', 'interpreter_output': {'target': 'blue_ceramic_mug', 'max_force_limit_n': 4.5, 'max_nav_velocity': 0.72}}
Belief: {'eva_position': 'living_room', 'room_busyness': 'medium', 'ambient_temp_c': 21.22, 'surface_friction': 0.4, 'battery_level_percent': 15.0}
Committed: {'action_approved': False, 'parameters': {'max_velocity_mps': 0.8, 'tilt_rate_radps': 0.2, 'contact_depth_m': 0.01}, 'warrant_verdict': {'passed': False, 'failed_criterion': 'safe', 'predictions': {'predicted_force_n': 9.45, 'predicted_wave_mm': 8.53, 'predicted_nav_velocity': 0.72}, 'warrant_checks': {'grounded': True, 'feasible': True, 'safe': False, 'energetically_viable': False, 'monitorable': True, 'repairable': True, 'recordable': True}}}
Observed: {'timestamp': 1793612743.359301, 'measured_temp_c': 21.22, 'measured_friction': 0.4, 'eva_position': 'living_room', 'room_busyness': 'medium'}

Entry 2
Asked: {'raw_utterance

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.
* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.

ANS: After analysing the Python code, I found that its overall structure follows the Integration Diagram discussed in the lecture. Each class is responsible for a particular stage of the robot's decision-making and execution process.

The mapping is as follows:
Integration Diagram Component	Corresponding Class / Method	Responsibility
Interpret & Ground	InterpretationSystem.interpret()	Converts the user's command into a structured goal specification.
Perceive	SensingSystem.sense() and PerceptionSystem.update()	Collects environmental information and updates the robot's belief state.
Propose	Proposer.propose()	Generates possible action parameters for completing the task.
Know & Predict	SemanticDigitalTwin.predict_properties()	Predicts the possible physical consequences of a proposed action.
Decide (Warrant Gate)	WarrantGate.validate()	Checks whether the proposed action satisfies the required safety and feasibility conditions.
Act & Regulate	ExecutionEngine.execute_action()	Executes the action after it has been approved.
Record & Justify	ExecutionEngine.record_trace()	Stores information about the request, belief state, decision and execution outcome.

Data flow during one cognitive decision cycle:

1. Eva gives the instruction: "Bring me my coffee fast!"
2. The Sensing System collects environmental information, such as temperature, friction and Eva's position.
3. The Perception System processes these observations and updates the robot's internal BeliefState.
4. The Interpretation System uses the instruction and updated belief to generate a structured GoalSpecification.
5. The Proposer generates candidate action parameters.
6. The Semantic Digital Twin predicts the physical consequences of the proposed action.
7. The Warrant Gate evaluates the predictions against the goal and system constraints.
8. If approved, the Executive executes the action. Otherwise, the proposal is rejected.
9. The Episode Recorder stores the request, belief, observations, decision and execution outcome for later auditing.

The Warrant Gate acts as a commitment boundary: a proposed action cannot proceed to physical execution unless it passes the required checks.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.

ANS: I modified the `SemanticDigitalTwin.predict_properties()` method to implement a dual-stage contact compliance model. In the first stage, the stiffness coefficient is calculated based on the surface friction coefficient, representing normal contact conditions. In the second stage, when the contact depth exceeds 0.02 m, the stiffness coefficient is multiplied by 1.8 to represent the increased resistance caused by bottoming out against a rigid internal fixture. The predicted contact force is then calculated using:

Predicted Force = Stiffness Coefficient × (Approach Velocity ^ 1.5) 

This modification allows the Semantic Digital Twin to account for increased contact resistance at greater contact depths and predict potentially unsafe contact forces before physical execution.

* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

ANS:I tested an aggressive contact proposal using an approach velocity of 0.35 m/s and a contact depth of 0.025 m. Since the contact depth exceeds the threshold of 0.02 m, the Semantic Digital Twin increases the stiffness coefficient by a factor of 1.8. Based on this calculation, the predicted contact force was 4.77 N, which exceeds the blue ceramic mug's maximum allowable force of 4.5 N.
As a result, the safety criterion failed, and the Warrant Gate rejected the proposal. The energy viability criterion also failed, meaning that the robot was not allowed to proceed with physical execution.From this experiment, I observed that the Warrant Gate evaluates the predicted consequences and resource requirements of an action before allowing the robot to execute it. This helps prevent potentially unsafe actions and ensures that the robot only proceeds when the required conditions are satisfied.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).

ANS:I modified the fluid dynamics model in the SemanticDigitalTwin.predict_properties() method to consider the effect of temperature on fluid viscosity. For this, I used a linear thermal degradation model with a reference temperature of 20°C and a thermal degradation coefficient of 0.01.

From my analysis, I observed that when the ambient temperature increases above the reference temperature, the effective viscosity of the fluid decreases. The model then uses this updated viscosity value to predict the liquid wave height. I also maintained a minimum viscosity value of 0.01 Pa·s to avoid zero or negative viscosity values during the calculation.

This modification helps the Semantic Digital Twin consider temperature changes while predicting fluid behaviour and possible sloshing during the pouring process.

* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

ANS: To analyse the effect of temperature on fluid sloshing, I performed a test by keeping the tilt rate constant at 0.15 rad/s and changing the ambient temperature.

The predicted results were:
Ambient temperature	Predicted wave height
20°C	6.26 mm
28°C	6.94 mm

I observed that the predicted wave height increased by 0.68 mm when the temperature increased from 20°C to 28°C.

This happens because higher temperatures reduce the effective viscosity of the fluid, allowing it to move more easily. As a result, the predicted wave height increases even though the tilt rate remains unchanged.

From this experiment, I understood that temperature is an important factor in predicting fluid stability. The Semantic Digital Twin considers this change so that the robot can evaluate the risk of excessive sloshing before executing the pouring action.

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.

ANS: I extended the Warrant Gate by introducing a seventh criterion called `energetically_viable`. From my analysis, this check ensures that the estimated battery drain for a proposed trajectory remains strictly below 15% of the total battery capacity and does not exceed the robot's currently available battery level.

The check uses the estimated battery drain stored in the `ParameterPacket` and the remaining battery level maintained in the `BeliefState`. This allows the robot to consider its energy availability before approving an action and prevents it from starting a task when sufficient battery power is not available.

* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

ANS: I introduced a low-battery condition in the `BeliefState` by setting the available battery level to 15%. I then tested a trajectory with an estimated battery drain of 16%.

During validation, the Warrant Gate rejected the proposal because the estimated energy requirement exceeded both the permitted 15% threshold and the robot's available battery level. As a result, the `energetically_viable` criterion returned `False`.

I also implemented a workflow condition so that when the energy viability check fails, the robot requests a recharge instead of proceeding with coffee delivery. After recharging, the system can check the battery level again before attempting the delivery.

Through this test, I demonstrated that the Warrant Gate does not simply evaluate whether an action is physically possible; it also considers whether the robot has sufficient resources to complete the task.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.

ANS: I implemented the `record_trace()` method inside the `ExecutionEngine` class to maintain an episodic audit trail of the robot's decision-making process.

Each audit entry stores important information, including:

- User request: The original command given to the robot and its interpreted goal.
- Belief: The robot's internal representation of the environment.
- Commitment decision: Whether the proposed action was approved or rejected by the Warrant Gate.
- Action parameters: The proposed movement velocity and tilt rate.
- Warrant verdict: The predicted physical outcomes and the results of the validation checks.
- Execution outcome: The actual contact force, fluid wave height, transferred volume and spill status, when physical execution takes place.

I also implemented the `audit_analyzer(audit_trail)` function to analyse the recorded information. It counts the approved and rejected proposals, identifies the most frequently failed Warrant Gate criterion and calculates the population variance of absolute force prediction errors whenever execution data is available.

In my current test, both proposals were rejected by the Warrant Gate. Since neither proposal was physically executed, no actual force measurements were available for comparison. Therefore, the force prediction error variance was reported as `None`.

This analysis helped me understand how an episodic audit trail can be used to investigate rejected actions and evaluate prediction accuracy when actual execution data is available.

* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

ANS: I extended the audit trail to maintain four separate records: `asked`, `belief`, `committed` and `observed`.

- Asked: Stores the original user instruction and the interpreted goal.
- Belief: Stores the robot's internal understanding of the current environment.
- Committed: Records the proposed action parameters and the decision made by the Warrant Gate.
- Observed: Stores the actual sensor measurements and execution outcomes.

From my understanding, maintaining these separate records is important because the robot's internal understanding and the actual state of the environment may not always be the same.

Pure data-driven imitation learning mainly learns to reproduce actions from previously observed demonstrations. However, it does not necessarily explain why a particular action was selected, what the robot believed about the environment or whether the action was safe under the current conditions.

By keeping these four records separate, we can compare the original request, the robot's beliefs, its decisions and the actual observations. This makes it easier to identify whether a problem occurred during interpretation, perception, decision-making or execution.

In my experiment, both proposals were rejected, but their predictions and failed Warrant Gate checks were still recorded. This shows that the system can maintain a trace of its decisions even when no physical execution takes place.

Conclusion: The audit trail improves traceability and accountability by preserving the information behind the robot's decisions. It also helps analyse failures and understand the difference between what the robot was asked to do, what it believed, what it decided and what actually happened.

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.


---

ANS: Part 3: Robot Anatomy

The diagram illustrates the main mechanical, sensing and actuation components of a mobile manipulator robot. These components allow the robot to move, perceive its surroundings and interact with objects in the environment.

**1. Links and Joints:**

Links are the rigid segments that form the robot's arms and body, while joints connect these segments and allow rotational or translational movement. Together, they provide the flexibility required to position the robot's arm and manipulate objects.

**2. Actuators and Sensors:**

Actuators, such as motors, generate movement in the robot's joints and wheels. Sensors collect information about the robot's internal state and external surroundings. This information supports perception, monitoring and decision-making.

**3. Effectors, Base, Torso and Head:**

- **Effectors:** These are the components that physically interact with objects in the environment. Grippers are examples of end-effectors.
- **Base:** The bottom platform supports the robot and contains its mobility system.
- **Torso:** The central body connects the base to the upper body and supports the robot's arms.
- **Head:** The upper section carries sensing equipment, such as cameras, to observe the surrounding environment.

**4. Wheels, Legs and Grippers:**

- **Wheels:** Allow the robot to move around its environment using a wheeled mobility system.
- **Legs:** Provide articulated movement and support locomotion in robots designed for leg-based mobility. The illustrated robot primarily uses a wheeled base.
- **Grippers:** Allow the robot to grasp, hold and manipulate objects during task execution.

**5. Proprioception, Exteroception and Interoception:**

- **Proprioception:** Provides information about the robot's own configuration and movement, such as joint positions and wheel rotation.
- **Exteroception:** Provides information about the external environment through sensors such as cameras and laser distance sensors.
- **Interoception:** Provides information about the robot's internal condition, such as battery level, motor temperature and system status.

**Connection to the Integrated Intelligent Systems Architecture:**

These physical components provide the foundation for sensing and acting in the Integrated Intelligent Systems architecture. Sensors collect information about the robot's body and its surroundings, which is used by the Sensing and Perception components to update the robot's understanding of the environment.

The Interpretation and Planning components determine the actions required to achieve the task, while the Semantic Digital Twin predicts their possible physical consequences. The Warrant Gate evaluates whether the proposed actions satisfy the required safety and feasibility conditions before execution.

Once an action is approved, the Acting component sends commands to the actuators, joints, wheels and grippers to perform the task. The resulting observations and execution outcomes can then be recorded for monitoring and future analysis.

Thus, the robot's anatomy provides the physical foundation for perception, mobility and object manipulation, allowing the intelligent architecture to interact with and perform tasks in the real world.

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 